## Init

In [0]:
from pyspark.sql import functions as F
from delta.tables import DeltaTable

In [0]:
%run /Workspace/Users/axl.dxn@gmail.com/atlikon_pipeline/1_setup/utilities

## Configure widgets

In [0]:
dbutils.widgets.text("catalog", "fmcg", "Catalog")
dbutils.widgets.text("data_source", "gross_price", "Data Source")

catalog = dbutils.widgets.get("catalog")
data_source = dbutils.widgets.get("data_source")

print(f"catalog: {catalog}, data_source: {data_source}")

## Configure S3 bucket path

In [0]:
base_path = f's3://atlikon-dp/{data_source}/*.csv'
print(f"base_path: {base_path}")

## Create DataFrame with raw data and metadata

In [0]:
df = (
    spark.read
    .format("csv")
    .option("header", True)
    .option("inferSchema", True)
    .load(base_path)
    .withColumn("read_timestamp", F.current_timestamp())
    .select("*", "_metadata.file_name", "_metadata.file_size")
)

## Sanity check of DataFrame

In [0]:
display(df.limit(10))

In [0]:
(
    df.write
    .format("delta")
    .option("delta.enableChangeDataFeed", "true")
    .mode("overwrite")
    .saveAsTable(f"{catalog}.{bronze_schema}.{data_source}")
)

## Sanity check of bronze table

In [0]:
query = f"SELECT * FROM {catalog}.{bronze_schema}.{data_source} LIMIT 10;"

df_check = spark.sql(query)

display(df_check.limit(10))